# AoA Estimation 

### Linear Antenna Arrays

In this task, our goal is to estimate the AoA of the reflected signal from the RFID tag. Consider the linear antenna array setup shown below.

<img src="media/image2.png" alt="drawing" width="600"/>

 Assuming that the channels measured at antenna element `Rxi` is given by `h_i` , where `1 ≤ i ≤ N` , the AoA of the signal can be given by:  

<img src="media/image.png" alt="drawing" width="600"/>


However, since RFID tags reflect signals, the above formula needs to be modified in order to estimate the AoA. You should write down the modified formula in your report, and explain it.


Next, you must implement the function `estimate_aoa` to estimate the AoA given the channel measurements hi, based on the modified formula. The `estimate_aoa` function should return AoA values in the resolution of 1 degree. This function will be evaluated on two data sets, simulated data and real data from RFID hardware. 



In [1]:
import numpy as np

## TODO: Implement the function estimate_aoa(h, d, wavelength) that estimates the angle of arrival of the signal
def estimate_aoa(h, d, wavelength):

    N = len(h)
    theta_scan = np.arange(0, 181, 1)   # resolution of 1 degree
    response = np.zeros(len(theta_scan), dtype=np.complex128)

    for theta_deg, theta in enumerate(theta_scan):
        theta_rad = np.deg2rad(theta)
        steering = np.exp(-1j * (np.arange(N) + 1) * 4 * np.pi / wavelength * d * np.cos(theta_rad))    # + 1 s.t. i = 1,..,N
        response[theta_deg] = np.abs(np.sum(np.exp(1j * np.deg2rad(h)) * steering))

    aoa = theta_scan[np.argmax(response)]
    
    return aoa


Testing on simulated data 

In [2]:
from utils import read_data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io as sio
import os

# Load simulated data from .mat file
data = sio.loadmat('Test/Test_AoA_Estimation.mat')

# Assuming 'h_channel', 'd', and 'lambda' are keys in the loaded data
h_channel = data['h_channel']
d = data['d'][0][0]
lambda_ = data['lambda'][0][0]

# Initialize angles array
angles = np.zeros(100)

# Calculate angles using estimate_aoa function
for k in range(100):
    h_curr = h_channel[k, :]
    angles[k] = estimate_aoa(h_curr, d, lambda_)

# Save angles array to a .mat file
sio.savemat('Results/AoA_estimation.mat', {'angles': angles})

In [3]:
# ### Debugging cell

# debug = sio.loadmat('Debugging_data/AoA_estimation_10.mat')
# print(f'Debug keys:\t', debug.keys())
# results = sio.loadmat('Results/AoA_estimation.mat')
# print(f'Results keys:\t', results.keys())

# debut_angles = debug['angles'].flatten()
# results_angles = results['angles'].flatten()
# print(f'\nDebug angles:\t', debut_angles)
# print(f'Results angles:\t', results_angles)

# results_cropped = results_angles[:len(debut_angles)]
# print(f'\nNumber of errors:\t', np.sum(np.abs(debut_angles - results_cropped) >= 0.5))

Testing on RFID data

In [4]:

# Read the real RFID data file
file = 'Lab_Data/lab3_task2.txt'
channel_log = read_data(file)

# Extract and process the data
ant = channel_log['ant'].values
freq = channel_log['Frequency'].iloc[0] * 1e3
pha = channel_log['Phase'].astype(float).values

phas = [None] * 4
h = np.zeros(4)

for k in range(4):
    phas[k] = pha[ant == (k + 1)]
    h[k] = np.mean(phas[k])

lambda_ = 3e8 / freq
d = lambda_ / 4


print(estimate_aoa(h, d, lambda_))



81


### Circular Antenna Arrays

In this subtask you will estimate the AoA from a circular array of RFIDs as shown below.

 <img src="media/image3.png" alt="drawing" width="600"/>

The circular array has n number of RFID tags and the radius of the array is R. The figure also shows the channels bk measured at antenna k for a signal arriving from θdirection. Based on
this, write down the formula similar to the equation provided above or calculating AoA for such a circular array. 

Assume the channels measured from RFID element i is given by hi . Note that your formula should account for the fact that RFID tags reflect signals, similar to the previous subpart. 

Now you must implement the function `estimate_aoa_circular` to estimate AoA given the channel measurements hi’s from a circular RFID array. 
The function also takes as input the radius of the array R and the λ. The `estimate_aoa_circular` function should return AoA values in the resolution of 1 degree. 

To evaluate your function, you must run the test script below which will save the output in the Results folder.

In [5]:
## TODO: Implement the following function
def estimate_aoa_circular(h, R, wavelength):
    
    N = len(h)
    theta_scan = np.arange(0, 360, 1)   # resolution of 1 degree
    response = np.zeros(len(theta_scan), dtype=np.complex128)

    phi = np.arange(N) * 2 * np.pi / N

    for theta_deg, theta in enumerate(theta_scan):
        theta_rad = np.deg2rad(theta)
        steering = np.exp(1j * 4 * np.pi / wavelength * R * np.cos(phi - theta_rad))
        response[theta_deg] = np.abs(np.sum(np.exp(1j * np.deg2rad(h)) * steering))

    aoa = theta_scan[np.argmax(response)]
    
    return aoa


Note you may have a one degree error that arises from the quantization of the values, which is expected. 

In [6]:
import numpy as np
import scipy.io as sio

# Load data from .mat file
data = sio.loadmat('Test/Test_AoA_Estimation_circular.mat')

# Assuming 'channels', 'R', and 'lambda' are keys in the loaded data
channels = data['channels']
R = data['R'][0][0]
lambda_ = data['lambda'][0][0]

# Initialize angles array
angles = np.zeros(100)

# Calculate angles using estimate_aoa_circular function
for k in range(100):
    h_curr = channels[k, :]
    angles[k] = estimate_aoa_circular(h_curr, R, lambda_)

# Save angles array to a .mat file
sio.savemat('Results/AoA_estimation_circular.mat', {'angles': angles})


In [7]:
# ### Debugging cell

# debug = sio.loadmat('Debugging_data/AoA_estimation_circular_10.mat')
# print(f'Debug keys:\t', debug.keys())
# results = sio.loadmat('Results/AoA_estimation_circular.mat')
# print(f'Results keys:\t', results.keys())

# debut_angles = debug['angles'].flatten()
# results_angles = results['angles'].flatten()
# print(f'\nDebug angles:\t', debut_angles)
# print(f'Results angles:\t', results_angles[:10])

# results_cropped = results_angles[:len(debut_angles)]
# print(f'\nNumber of errors:\t', np.sum(np.abs(debut_angles - results_cropped) >= 0.5))